In [0]:
USE CATALOG dbw_kroger_insights;

In [0]:
CREATE OR REPLACE TABLE gold.ic_department_metrics
COMMENT 'Instacart department metrics for dashboards. One row per department. Penetration and item share over all orders; reorder rate excludes first orders; relative cart position 0 = first item, 1 = last.'
AS
WITH totals AS (
    SELECT COUNT(*) AS total_orders FROM gold.ic_dim_order
),

-- 1. Penetration + item share (from Module 4, Q1)
penetration AS (
    SELECT
        p.department,
        COUNT(DISTINCT f.order_id)  AS orders_with_department,
        COUNT(*)                    AS items
    FROM gold.ic_fact_order_products f
    JOIN gold.ic_dim_product p ON f.product_id = p.product_id
    GROUP BY p.department
),
-- 2. Reorder rate, excluding first orders (Module 4, Q2)
reorder AS (
    SELECT
        p.department,
        ROUND(100.0 * AVG(f.reordered), 2)          AS reorder_rate_pct
    FROM gold.ic_fact_order_products f
    JOIN gold.ic_dim_order   o ON f.order_id   = o.order_id
    JOIN gold.ic_dim_product p ON f.product_id = p.product_id
    WHERE NOT o.is_first_order
    GROUP BY p.department
),

-- 3. Relative cart position (Module 4, Q3)
cart_position AS (
    SELECT
        p.department,
        ROUND(AVG(
            (b.add_to_cart_order - 1) / NULLIF(b.basket_size - 1, 0)
        ), 4)                                       AS avg_relative_cart_position
    FROM (
        SELECT
            product_id,
            add_to_cart_order,
            COUNT(*) OVER (PARTITION BY order_id)   AS basket_size
        FROM gold.ic_fact_order_products
    ) b
    JOIN gold.ic_dim_product p ON b.product_id = p.product_id
    GROUP BY p.department
)

SELECT
    pe.department,
    pe.orders_with_department,
    ROUND(100.0 * pe.orders_with_department / t.total_orders, 2)   AS order_penetration_pct,
    pe.items,
    ROUND(100.0 * pe.items / SUM(pe.items) OVER (), 2)             AS item_share_pct,
    ro.reorder_rate_pct,
    cp.avg_relative_cart_position
FROM penetration pe
CROSS JOIN totals t
LEFT JOIN reorder       ro ON pe.department = ro.department
LEFT JOIN cart_position cp ON pe.department = cp.department;


In [0]:
SELECT * FROM gold.ic_department_metrics ORDER BY order_penetration_pct DESC;

In [0]:
CREATE OR REPLACE TABLE gold.ic_top_pairs
COMMENT 'Top 50 product pairs by lift among the 100 most-ordered Instacart products (pairs with >= 1,000 co-occurring orders). Support, confidence (A to B) and lift.'
AS
WITH top_100_products AS (
    SELECT product_id
    FROM gold.ic_fact_order_products
    GROUP BY product_id
    ORDER BY COUNT(DISTINCT order_id) DESC
    LIMIT 100
),
order_items AS (
    SELECT DISTINCT f.order_id, f.product_id
    FROM gold.ic_fact_order_products f
    JOIN top_100_products t ON f.product_id = t.product_id
),
product_order_counts AS (
    SELECT product_id, COUNT(*) AS order_count
    FROM order_items
    GROUP BY product_id
),
total_orders_cte AS (
    SELECT COUNT(*) AS total_orders FROM gold.ic_dim_order
),
pairs AS (
    SELECT a.product_id AS product_a, b.product_id AS product_b, COUNT(*) AS pair_orders
    FROM order_items a
    JOIN order_items b
      ON a.order_id = b.order_id
     AND a.product_id < b.product_id
    GROUP BY a.product_id, b.product_id
    HAVING COUNT(*) >= 1000
)
SELECT
    p.product_a,
    pa.product_name                                              AS product_a_name,
    pa.aisle                                                     AS aisle_a,
    p.product_b,
    pb.product_name                                              AS product_b_name,
    pb.aisle                                                     AS aisle_b,
    p.pair_orders,
    ROUND(100.0 * p.pair_orders / t.total_orders, 4)             AS support_pct,
    ROUND(100.0 * p.pair_orders / ca.order_count, 2)             AS confidence_a_to_b_pct,
    ROUND( (p.pair_orders * 1.0 / t.total_orders)
         / ((ca.order_count * 1.0 / t.total_orders) * (cb.order_count * 1.0 / t.total_orders)), 2) AS lift
FROM pairs p
CROSS JOIN total_orders_cte t
JOIN product_order_counts ca ON p.product_a = ca.product_id
JOIN product_order_counts cb ON p.product_b = cb.product_id
JOIN gold.ic_dim_product  pa ON p.product_a = pa.product_id
JOIN gold.ic_dim_product  pb ON p.product_b = pb.product_id
ORDER BY lift DESC
LIMIT 50;

In [0]:
SELECT product_a_name, product_b_name, pair_orders, lift
FROM gold.ic_top_pairs
ORDER BY lift DESC
LIMIT 10;